# DCN-v2 Reject Inference Pipeline (Phase 8.7)

**Purpose**: Implement Teacher-Student training loop with Isolation Forest filtering.

**Methodology**:
1. **Teacher Inference**: Generate embeddings and predictions for Rejected (D_R) population.
2. **Uncertainty Filtering**: Train Isolation Forest on Approved (D_L) embeddings to filter OOD rejects.
3. **Pseudo-Labeling**: Assign labels to filtered D_R samples using Teacher predictions.
4. **Student Training**: Train new DCN-v2 on Augmented Dataset (D_L + D_R_pseudo).
5. **Evaluation**: Compare Diamond Recovery (Precision@20%) on D_R ground truth.

**Protocol**: Pair-programming approved (2026-02-15)

---

## Cell 1: Setup & Imports

Mount Google Drive and import required libraries.

In [ ]:
# Purpose: Mount Google Drive and set up environment
# Why: Access to persisted data and models
# Approved: 2026-02-15

from google.colab import drive
drive.mount('/content/drive')

# Set project path (adjust to your Drive folder)
PROJECT_PATH = '/content/drive/MyDrive/final_ex'  # <-- EDIT THIS

import sys
sys.path.insert(0, f'{PROJECT_PATH}/src')

In [ ]:
# Purpose: Import libraries
# Why: Standard stack + DCN-v2 + Isolation Forest
# Approved: 2026-02-15

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.ensemble import IsolationForest
from typing import Dict, List, Tuple
import os
import json
import matplotlib.pyplot as plt

# DCN-v2 modules
from models.dcn_v2 import DCNv2, compute_quantiles_for_binning

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 2: Load Data & Teacher Model

Load prepared dataframes and the best checkpoint from Phase 8.6.

In [ ]:
# Purpose: Load DataFrames
# Why: Need D_L (Train/Val/Test) for baselines and D_R (Reject) for inference
# Approved: 2026-02-15

DATA_PATH = f'{PROJECT_PATH}/data/processed'
MODELS_PATH = f'{PROJECT_PATH}/outputs/models'
TEACHER_PATH = f'{MODELS_PATH}/dcn_v2_best.pth'

# Load D_L (Approved)
X_train = pd.read_parquet(f'{DATA_PATH}/vime_X_train_fixed.parquet')
X_val = pd.read_parquet(f'{DATA_PATH}/vime_X_val_fixed.parquet')
X_test = pd.read_parquet(f'{DATA_PATH}/vime_X_test_fixed.parquet')
y_train = pd.read_parquet(f'{DATA_PATH}/vime_y_train.parquet')
y_val = pd.read_parquet(f'{DATA_PATH}/vime_y_val.parquet')
y_test = pd.read_parquet(f'{DATA_PATH}/vime_y_test.parquet')

# Load D_R (Rejected)
# Note: using vime_X_reject_fixed.parquet which has same features as D_L
X_reject = pd.read_parquet(f'{DATA_PATH}/vime_X_reject_fixed.parquet')
# Load Ground Truth for D_R (ONLY for final evaluation, NOT for training)
df_reject_gt = pd.read_parquet(f'{DATA_PATH}/D_R_ground_truth.parquet')
y_reject_gt = df_reject_gt['TARGET']

NUMERICAL_FEATURES = X_train.columns.tolist()
print(f'D_L Train: {X_train.shape}, D_R Only: {X_reject.shape}')

In [ ]:
# Purpose: Re-define components for DataLoader
# Why: Need to replicate training setup for inference
# Approved: 2026-02-15

class DCNv2Dataset(Dataset):
    def __init__(self, X_df, y_df=None, feature_names=None):
        self.X = X_df.values.astype(np.float32)
        # Handle unlabeled data (D_R)
        if y_df is not None:
            self.y = y_df.values.astype(np.float32).flatten()
        else:
            self.y = None
        self.feature_names = feature_names
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        row = self.X[idx]
        numerical_values = {name: torch.tensor(row[i]) for i, name in enumerate(self.feature_names)}
        if self.y is not None:
            return numerical_values, torch.tensor(self.y[idx])
        else:
            return numerical_values

def dcnv2_collate_fn(batch):
    # Check if batch contains labels or just features
    has_labels = isinstance(batch[0], tuple)
    
    if has_labels:
        numerical_dicts, labels = zip(*batch)
    else:
        numerical_dicts = batch
        labels = None
    
    feature_names = numerical_dicts[0].keys()
    batched_numerical = {
        name: torch.stack([d[name] for d in numerical_dicts])
        for name in feature_names
    }
    
    if labels is not None:
        batched_labels = torch.stack(labels).unsqueeze(1)
        return batched_numerical, batched_labels
    else:
        return batched_numerical

In [ ]:
# Purpose: Load Teacher Model
# Why: Need pre-trained model to generate pseudo-labels
# Approved: 2026-02-15

checkpoint = torch.load(TEACHER_PATH)
# Load hyperparameters from results JSON to ensure consistency
with open(f'{PROJECT_PATH}/outputs/results/dcn_v2_results.json', 'r') as f:
    results_dcn = json.load(f)
    hp = results_dcn['hyperparameters']

teacher_model = DCNv2(
    numerical_features=NUMERICAL_FEATURES,
    categorical_cardinalities={},
    num_bins=hp['num_bins'],
    numerical_embed_dim=hp['numerical_embed_dim'],
    cross_layers=hp['cross_layers'],
    cross_rank=hp['cross_rank'],
    deep_hidden=hp['deep_hidden'],
    dropout=hp['dropout']
)

teacher_model.load_state_dict(checkpoint['model_state_dict'])
teacher_model = teacher_model.to(device)
teacher_model.eval()

# Access SoftBinning cutpoints to verify they are loaded
print("Teacher model loaded. Verifying cutpoints...")
print(teacher_model.embedding.numerical_binning[NUMERICAL_FEATURES[0]].cutpoints.data[:5])

## Cell 3: Embeddings & Inference on D_R

Generate 4048-dim embeddings ($x_0$ or cross output) for D_R to use in Isolation Forest.

In [ ]:
# Purpose: Extract embeddings and predictions
# Why: Embeddings (x0) needed for IF; Predictions needed for pseudo-labeling
# Approved: 2026-02-15

def get_embeddings_and_preds(model, loader, device):
    model.eval()
    all_embeddings = []
    all_preds = []
    
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, tuple):
                numerical_values = batch[0]
            else:
                numerical_values = batch
                
            numerical_values = {k: v.to(device) for k, v in numerical_values.items()}
            
            # Get x0 embedding (output of EmbeddingEngine)
            x0 = model.embedding(numerical_values, {})
            
            # Get predictions
            probs = model(numerical_values, {})
            
            all_embeddings.append(x0.cpu().numpy())
            all_preds.append(probs.cpu().numpy())
            
    return np.vstack(all_embeddings), np.vstack(all_preds).flatten()

BATCH_SIZE = 1024

# Create loaders
train_loader_if = DataLoader(DCNv2Dataset(X_train, feature_names=NUMERICAL_FEATURES), batch_size=BATCH_SIZE, shuffle=False, collate_fn=dcnv2_collate_fn)
reject_loader_if = DataLoader(DCNv2Dataset(X_reject, feature_names=NUMERICAL_FEATURES), batch_size=BATCH_SIZE, shuffle=False, collate_fn=dcnv2_collate_fn)

print("Generating embeddings for D_L (Train)...")
emb_train, _ = get_embeddings_and_preds(teacher_model, train_loader_if, device)

print("Generating embeddings/preds for D_R (Reject)...")
emb_reject, preds_reject = get_embeddings_and_preds(teacher_model, reject_loader_if, device)

print(f"Train Embeddings: {emb_train.shape}")
print(f"Reject Embeddings: {emb_reject.shape}")

## Cell 4: Uncertainty Filtering (Isolation Forest)

Train IF on approved embeddings, filter rejected applicants.

In [ ]:
# Purpose: Train Isolation Forest and filter D_R
# Why: Remove "Out of Distribution" rejects that the teacher cannot reliably score
# Approved: 2026-02-15

CONTAMINATION = 0.1  # Approx 10% expected outliers in training, loose heuristic

print(f"Training Isolation Forest on {len(emb_train)} approved samples...")
iso_forest = IsolationForest(
    n_estimators=100, 
    max_samples='auto', 
    contamination='auto', 
    random_state=42, 
    n_jobs=-1
)
iso_forest.fit(emb_train)

# Predict on D_R
reject_is_inlier = iso_forest.predict(emb_reject) # 1 = inlier, -1 = outlier
reject_scores = iso_forest.score_samples(emb_reject)

mask_inlier = reject_is_inlier == 1
n_inliers = mask_inlier.sum()
pct_kept = 100 * n_inliers / len(X_reject)

print(f"Inliers kept: {n_inliers}/{len(X_reject)} ({pct_kept:.1f}%)")

# Visualize anomaly scores
plt.figure(figsize=(10, 5))
plt.hist(reject_scores, bins=50, label='Rejected', alpha=0.5)
plt.title('Isolation Forest Anomaly Scores (Higher is better)')
plt.show()

## Cell 5: Pseudo-Labeling & Augmentation

Create combined dataset for Student training.

In [ ]:
# Purpose: Generate pseudo-labels and combine datasets
# Why: Student learns from both real labels and teacher's knowledge of D_R
# Approved: 2026-02-15

# Filter D_R
X_reject_filtered = X_reject[mask_inlier].copy()
preds_reject_filtered = preds_reject[mask_inlier]

# Pseudo-labeling (Hard Threshold)
# Aiming for ~30% default rate consistent with Route B calibration
PSEUDO_THRESHOLD = np.quantile(preds_reject_filtered, 0.70) # Top 30% are defaults
print(f"Pseudo-label threshold (Top 30%): {PSEUDO_THRESHOLD:.4f}")

y_reject_pseudo = (preds_reject_filtered >= PSEUDO_THRESHOLD).astype(np.float32)
print(f"Pseudo Default Rate: {y_reject_pseudo.mean():.1%}")

# Augment Dataset
X_combined = pd.concat([X_train, X_reject_filtered], axis=0).reset_index(drop=True)
y_combined = np.concatenate([y_train.values.flatten(), y_reject_pseudo], axis=0)

print(f"Augmented Dataset Shape: {X_combined.shape}, Labels: {y_combined.shape}")

# Create Student Dataloader
# Converting y_combined to DataFrame to match Dataset init
y_combined_df = pd.DataFrame(y_combined, columns=['TARGET'])
student_dataset = DCNv2Dataset(X_combined, y_combined_df, NUMERICAL_FEATURES)
student_loader = DataLoader(
    student_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=dcnv2_collate_fn, 
    num_workers=2
)

## Cell 6: Train Student Model

Train fresh DCN-v2 on augmented data.

In [ ]:
# Purpose: Instantiate and Train Student
# Why: Student has "seen" the rejected population
# Approved: 2026-02-15

student_model = DCNv2(
    numerical_features=NUMERICAL_FEATURES,
    categorical_cardinalities={},
    num_bins=hp['num_bins'],
    numerical_embed_dim=hp['numerical_embed_dim'],
    cross_layers=hp['cross_layers'],
    cross_rank=hp['cross_rank'],
    deep_hidden=hp['deep_hidden'],
    dropout=hp['dropout']
)
student_model = student_model.to(device)

# Loss with UPDATED pos_weight for augmented dataset
n_pos = y_combined.sum()
n_neg = len(y_combined) - n_pos
pos_weight_student = torch.tensor([n_neg / n_pos], device=device)
print(f'Student pos_weight: {pos_weight_student.item():.2f}')

criterion_student = nn.BCEWithLogitsLoss(pos_weight=pos_weight_student)
optimizer_student = torch.optim.AdamW(student_model.parameters(), lr=1e-3, weight_decay=1e-5)

# Quantiles for Student SoftBinning
# Re-computing on combined data is better, but training data quantiles are acceptable approx
# Let's reuse training quantiles to avoid re-computation overhead for now (or could optimize)
# Using Teacher's cutpoints initialization logic
quantiles_dict = {}
for feat in NUMERICAL_FEATURES:
    values = X_train[feat].values # Using original train for stability
    quantiles = compute_quantiles_for_binning(torch.tensor(values), hp['num_bins'])
    quantiles_dict[feat] = quantiles

for feat in NUMERICAL_FEATURES:
    student_model.embedding.numerical_binning[feat].cutpoints.data = quantiles_dict[feat].to(device)

# Copy-paste training loop helper from previous notebook (or import if modularized)
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for numerical_values, labels in loader:
        numerical_values = {k: v.to(device) for k, v in numerical_values.items()}
        labels = labels.to(device)
        
        x0 = model.embedding(numerical_values, {})
        cross_out = model.cross_network(x0)
        deep_out = model.deep_network(x0)
        combined = torch.cat([cross_out, deep_out], dim=1)
        logits = model.head(combined)
        
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for numerical_values, labels in loader:
            numerical_values = {k: v.to(device) for k, v in numerical_values.items()}
            probs = model(numerical_values, {})
            all_preds.extend(probs.cpu().numpy().flatten())
            all_labels.extend(labels.numpy().flatten())
    return roc_auc_score(all_labels, all_preds)

# Reuse Val Loader from Cell 2
val_loader = DataLoader(DCNv2Dataset(X_val, y_val, NUMERICAL_FEATURES), batch_size=BATCH_SIZE, shuffle=False, collate_fn=dcnv2_collate_fn)

# Train Loop
N_EPOCHS = 50 # Maybe fewer needed for student
best_val_auc = 0.0
STUDENT_PATH = f'{MODELS_PATH}/dcn_v2_student.pth'

for epoch in range(N_EPOCHS):
    loss = train_epoch(student_model, student_loader, criterion_student, optimizer_student, device)
    val_auc = evaluate(student_model, val_loader, device)
    print(f'Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f}')
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(student_model.state_dict(), STUDENT_PATH)
        print("New Best Student!")

## Cell 7: Final Evaluation (Diamond Recovery)

Compare Teacher vs Student on D_R Ground Truth.

In [ ]:
# Purpose: Evaluate Diamond Recovery
# Why: Measure business impact (Approved: 2026-02-15)

# Load best student
student_model.load_state_dict(torch.load(STUDENT_PATH))

# Create Loader for D_R with Ground Truth Labels
reject_gt_dataset = DCNv2Dataset(X_reject, pd.DataFrame(y_reject_gt), NUMERICAL_FEATURES)
reject_gt_loader = DataLoader(reject_gt_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=dcnv2_collate_fn)

def get_preds(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for numerical_values, _ in loader:
            numerical_values = {k: v.to(device) for k, v in numerical_values.items()}
            probs = model(numerical_values, {})
            preds.extend(probs.cpu().numpy().flatten())
    return np.array(preds)

preds_teacher = get_preds(teacher_model, reject_gt_loader)
preds_student = get_preds(student_model, reject_gt_loader)

def calc_diamond_metrics(y_true, y_pred, name):
    auc_score = roc_auc_score(y_true, y_pred)
    # Diamond = Good Borrower (Label=0). Model predicts Default (Label=1).
    # We want low scores for diamonds.
    # Precision@20%: Identify the "safest" 20% of rejects.
    # Sort by risk (ascending) -> took top k%
    
    df = pd.DataFrame({'target': y_true, 'prob': y_pred})
    df.sort_values('prob', ascending=True, inplace=True) # Lowest risk first
    
    k = 0.20
    top_k = df.head(int(len(df) * k))
    n_diamonds = (top_k['target'] == 0).sum()
    precision = n_diamonds / len(top_k)
    
    print(f"--- {name} ---")
    print(f"AUC on D_R: {auc_score:.4f}")
    print(f"Precision@20% (Diamonds): {precision:.1%}")
    print(f"Diamonds Recovered: {n_diamonds}")
    return n_diamonds

print("Ground Truth Analysis (D_R):")
diamonds_teacher = calc_diamond_metrics(y_reject_gt, preds_teacher, "Teacher")
diamonds_student = calc_diamond_metrics(y_reject_gt, preds_student, "Student")

print(f"\nStudent Lift: {diamonds_student - diamonds_teacher:+d} diamonds")